### Script setup

In [2]:
from pv_chip_aarhus.analysis.data_io import DataIO
from pv_chip_aarhus.analysis.analysis_params import dataset_dir  # refer to your config file to
    # instead of hardcoding the paths in each script

# Initiate dataloader
data_io = DataIO(dataset_dir)

# Print available sessions
print(f'Sessions in datadir:')
for sid in data_io.sessions:
    print(f'\t{sid}')

Sessions in datadir:
	test_data


### Load data

In [3]:
# Load a session
data_io.load_session('test_data')

### Access tables with metadata
All tables are pandas dataframes. These are the attributes I used all the time:

`df.index.values`: the index for each row, this can be used to quickly access a row

`df.loc[INDEX]`: this is how to index into a row

`df.columns`: list available columns

`df.loc[INDEX, COLUMN]` : gives value of a single cell

`df.query("COLUMN == VALUE")` : filters the dataframe based on column values

`df.iterrows()`  : allow you to loop over  each row of the table

In [6]:
# Inspect the recording dataframe
print(data_io.recording_df.columns)

# Lets print some data about this session
for row_index, row_values in data_io.recording_df.iterrows():
    print(f'{row_index}, {row_values["stim_file"]}: stimsource = {row_values["stimsource"]}')


Index(['trigger_file', 'stim_file', 'lasermode', 'stimsource', 'laser_ch',
       'varied_param'],
      dtype='object')
0, 01_pow_N_P_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = P
1, 02_pow_N_L_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = L
2, 03_pow_LC_B_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = B
3, 04_dur_N_P_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = P
4, 05_dur_N_L_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = L
5, 06_dur_LC_B_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = B
6, 07_del_N_B_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = B
7, 08_del2_N_B_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = B
8, 09_flick_N_L_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = L
9, 10_longstim_N_P_04-06-2026_ch8_eye1_2nd_half.dat: stimsource = P


In [10]:
# Other tables we have are the cluster dataframe and the trial dataframe
data_io.cluster_df.head()
data_io.trial_df.head()

,stimsource,rec_nr,pchr_delay,pchr_on_duration,pchr_off_duration,pchr_repeats,varied_param,pchr_power,iti,laser_delay,laser_on_duration,laser_off_duration,laser_repeats,laser_power,train_id
0,P,1.0,0.0,20.0,480.0,30.0,pow,12.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_000_01
1,P,1.0,0.0,20.0,480.0,30.0,pow,25.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_001_01
2,P,1.0,0.0,20.0,480.0,30.0,pow,50.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_002_01
3,P,1.0,0.0,20.0,480.0,30.0,pow,100.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_003_01
4,L,2.0,NaN,NaN,NaN,NaN,pow,NaN,10000.0,0.0,30.0,470.0,30.0,999.0,tid_004_02


In [14]:
# The most important ones are the busrt dataframe, which contain the sitmulation onsets, and the spiketimes dict, which contains the spiketimes.

# spiketimes are stored per recording.
print(data_io.spiketimes.keys())

# The burst dataframe is used to get the stimulus onsets
data_io.burst_df.head()

dict_keys(['01_N_P_pow', '02_N_L_pow', '03_LC_B_pow', '04_N_P_dur', '05_N_L_dur', '06_LC_B_dur', '07_N_B_del', '08_N_B_del2', '10_N_P_longstim'])


,pchr_onset,pchr_offset,stimsource,rec_nr,pchr_delay,pchr_on_duration,pchr_off_duration,pchr_repeats,varied_param,pchr_power,iti,laser_delay,laser_on_duration,laser_off_duration,laser_repeats,laser_power,train_id,rec_id,laser_onset,laser_offset
0,5969.250000,5989.299805,P,1.0,0.0,20.0,480.0,30.0,pow,12.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_000_01,01_N_P_pow,NaN,NaN
1,6469.250000,6489.350098,P,1.0,0.0,20.0,480.0,30.0,pow,12.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_000_01,01_N_P_pow,NaN,NaN
2,6969.350098,6989.450195,P,1.0,0.0,20.0,480.0,30.0,pow,12.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_000_01,01_N_P_pow,NaN,NaN
3,7469.450195,7489.500000,P,1.0,0.0,20.0,480.0,30.0,pow,12.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_000_01,01_N_P_pow,NaN,NaN
4,7969.500000,7989.600098,P,1.0,0.0,20.0,480.0,30.0,pow,12.0,10000.0,NaN,NaN,NaN,NaN,NaN,tid_000_01,01_N_P_pow,NaN,NaN


### Example of how to load all data for the 'pow' sequence

In [30]:
import numpy as np
varied_param = 'pow'
stimsource = 'P'
burst_df = data_io.burst_df.query('varied_param == @varied_param and stimsource == @stimsource')
print(f'found {burst_df.shape[0]} stimulation onsets in {len(burst_df["train_id"].unique())} trials')

# Use the groupby function, to get stimulation onsets for each trial
all_spiketimes = {}
for train_id, train_df in burst_df.groupby('train_id'):

    # Each row should have the same stimulation parameter. Normally you dont have to check this
    # but here im checking that anyway
    assert len(train_df['pchr_power'].unique()) == 1
    pchr_power = train_df.iloc[0]['pchr_power']  # the iloc function returns the row based on row number, not index
        # like with the loc function
    rec_id = train_df.iloc[0]['rec_id']  # we need this to get the spiketimes
    print(f'{train_id}: polychrome power: {pchr_power}')

    # Take the first cluster in this recording, in the future we should use cluster_df to point to the cells
    cluster_id = list(data_io.spiketimes[rec_id].keys())[0]
    spiketrain = data_io.spiketimes[rec_id][cluster_id]

    # Set interval from which to extract the spiketimes
    t_pre = 100
    t_post = 200

    spiketimes_trial = []  # this will now contain all the spiketimes
    for burst_i, burst_info in train_df.iterrows():
        onset = burst_info.pchr_onset
        idx = np.where((spiketrain >= onset - t_pre) & (spiketrain <= onset + t_post))[0]
        spiketimes_burst = spiketrain[idx] - onset  # normalize to stimulus onset
        spiketimes_trial.append(spiketimes_burst)

    all_spiketimes[train_id] = spiketimes_trial

found 120 stimulation onsets in 4 trials
tid_000_01: polychrome power: 12.0
tid_001_01: polychrome power: 25.0
tid_002_01: polychrome power: 50.0
tid_003_01: polychrome power: 100.0


### Plot the spiketrain

In [51]:
import plotly.graph_objects as go

fig = go.Figure()

# Setup variables for plotting
burst_offset   = 0
x_plot, y_plot = [], []
x_lines_laser, y_lines_laser = [], []
x_lines_dmd, y_lines_dmd = [], []

yticks         = []
ytext          = []

for train_id, spiketrains in all_spiketimes.items():
    df = data_io.trial_df.query('train_id == @train_id')
    stim_duration = df.iloc[0]['pchr_on_duration']
    pchr_power = df.iloc[0]['pchr_power']

    print(f'{train_id}, stimduration: {stim_duration:.0f}, power: {pchr_power}')

    for burst_i, sp in enumerate(spiketrains):
        x_plot.append(np.vstack([sp, sp, np.full(sp.size, np.nan)]).T.flatten())
        y_plot.append(np.vstack([np.ones(sp.size) * burst_offset,
                                np.ones(sp.size)* burst_offset +1, np.full(sp.size, np.nan)]).T.flatten())
        burst_offset += 1

x_plot = np.hstack(x_plot)
y_plot = np.hstack(y_plot)

fig.add_scatter(
    x=x_plot, y=y_plot,
    mode='lines', line=dict(width=5, color='white'),
    showlegend=False,
)
fig.show()


tid_000_01, stimduration: 20, power: 12.0
tid_001_01, stimduration: 20, power: 25.0
tid_002_01, stimduration: 20, power: 50.0
tid_003_01, stimduration: 20, power: 100.0
